In [13]:
import pandas as pd

DEMO_FOLDER = "data_latihan"   # folder data_latihan (di sebelah repo)

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(train.shape, test.shape)

(52889, 18) (22665, 17)


In [14]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52889 entries, 0 to 52888
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                52889 non-null  object
 1   sumber            52889 non-null  object
 2   kodekat           52889 non-null  object
 3   kodesubkat        52889 non-null  object
 4   kategori          52889 non-null  object
 5   subkategori       52889 non-null  object
 6   katakunci         52889 non-null  object
 7   tanggal           52889 non-null  object
 8   judul_berita      52889 non-null  object
 9   konten_berita     52889 non-null  object
 10  nama_tokoh        52889 non-null  object
 11  jabatan           52889 non-null  object
 12  organisasi        52889 non-null  object
 13  lokasi            52889 non-null  object
 14  alias             52889 non-null  object
 15  kutipan           52889 non-null  object
 16  sentimen_kutipan  52889 non-null  object
 17  sentimen_ber

In [15]:
import os

import numpy as np
import torch
from transformers import AutoTokenizer

PRETRAINED_FOLDER = "pretrained"   # UBAH
MODEL_NAME = "indobert"            # UBAH
TEXT = "konten_berita"                      # UBAH
TARGET = "sentimen_berita"                   # UBAH
MAX_LENGTH = 128                   # UBAH: lihat persentil di bawah
BATCH_SIZE = 16                    # UBAH
EPOCHS = 4                         # UBAH: 3–5 biasanya cukup
LEARNING_RATE = 3e-5               # UBAH: 1e-5 .. 5e-5
PATIENCE = 2                       # berhenti kalau F1 val tidak naik 2 epoch berturut-turut
SEED = 42                          # UBAH
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda"        # UBAH: False kalau GPU lama / error bf16
os.makedirs("outputs", exist_ok=True)
torch.manual_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(
    f"../{PRETRAINED_FOLDER}/{MODEL_NAME}")
train[TEXT] = train[TEXT].fillna("").astype(str)
test[TEXT] = test[TEXT].fillna("").astype(str)
token_counts = []
for text in train[TEXT].head(2000):
    token_counts.append(len(tokenizer(text)["input_ids"]))
print("token per teks, persentil 50/95/99:", np.percentile(token_counts, [50, 95, 99]))

token per teks, persentil 50/95/99: [ 462.5  1022.05 1666.11]


In [16]:
import torch

# 1. Cek apakah CUDA tersedia
cuda_tersedia = torch.cuda.is_available()
print(f"CUDA Tersedia: {cuda_tersedia}")

if cuda_tersedia:
    # 2. Cek jumlah GPU yang terdeteksi
    print(f"Jumlah GPU: {torch.cuda.device_count()}")

    # 3. Cek nama GPU yang sedang digunakan
    print(f"Nama GPU: {torch.cuda.get_device_name(0)}")

    # 4. Tes menjalankan tensor di GPU
    x = torch.rand(3, 3).cuda()
    print("\nTes Tensor di GPU berhasil:")
    print(x)


CUDA Tersedia: True
Jumlah GPU: 1
Nama GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU

Tes Tensor di GPU berhasil:
tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009],
        [0.2566, 0.7936, 0.9408]], device='cuda:0')


In [17]:
from torch.utils.data import Dataset

CLASSES = sorted(train[TARGET].unique())
train["target"] = train[TARGET].map(dict(zip(CLASSES, range(len(CLASSES)))))
test["target"] = 0                                # pengisi, tidak dipakai


class TextDataset(Dataset):
    def __init__(self, table):
        self.tokens = tokenizer(table[TEXT].tolist(), truncation=True, max_length=MAX_LENGTH)
        self.targets = table["target"].tolist()

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        item = {"input_ids": self.tokens["input_ids"][i],
                "attention_mask": self.tokens["attention_mask"][i]}
        item["labels"] = self.targets[i]
        return item


print(TextDataset(train.head(2))[0])

{'input_ids': [2, 4408, 30464, 644, 30465, 30469, 9281, 10380, 4318, 664, 30464, 16158, 30465, 126, 9648, 4049, 1147, 30464, 14656, 30465, 5397, 27519, 4476, 30356, 10276, 30370, 34, 234, 9921, 1578, 27843, 988, 393, 19, 3254, 2264, 14656, 501, 3921, 2077, 2906, 245, 22062, 79, 5731, 4522, 262, 6189, 30470, 30458, 2195, 9921, 3693, 339, 1456, 41, 8796, 9010, 464, 5053, 4198, 3921, 339, 8227, 30468, 11436, 4198, 776, 4522, 262, 30468, 30458, 661, 4887, 2120, 13796, 30357, 8687, 4152, 4235, 1697, 112, 19424, 34, 18248, 26, 4408, 30468, 3253, 30470, 19424, 256, 18248, 4887, 2120, 112, 5842, 13047, 7317, 26, 4486, 28769, 4408, 30470, 6810, 4887, 186, 11436, 4198, 10782, 1835, 600, 27341, 1633, 7147, 7513, 2011, 823, 23331, 41, 3574, 988, 6680, 1835, 600, 30377, 30470, 3], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [18]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

part_train, part_val = train_test_split(train, test_size=0.2, stratify=train["target"],
                                        random_state=42)
collate = DataCollatorWithPadding(tokenizer)
train_loader = DataLoader(TextDataset(part_train), batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate)
val_loader = DataLoader(TextDataset(part_val), batch_size=BATCH_SIZE * 2, collate_fn=collate)
test_loader = DataLoader(TextDataset(test), batch_size=BATCH_SIZE * 2, collate_fn=collate)
print(len(CLASSES), "kelas |", len(part_train), "train |", len(part_val), "val")

3 kelas | 42311 train | 10578 val


In [19]:
import numpy as np
import torch
from torch import nn
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

network = AutoModelForSequenceClassification.from_pretrained(
    f"../{PRETRAINED_FOLDER}/{MODEL_NAME}", num_labels=len(CLASSES),
    id2label=dict(zip(range(len(CLASSES)), CLASSES)))
network = network.to(DEVICE)
loss_function = nn.CrossEntropyLoss()
# counts = np.bincount(part_train["target"], minlength=len(CLASSES))          # kelas timpang
# class_weight = torch.tensor(len(part_train) / (len(CLASSES) * counts), dtype=torch.float)
# loss_function = nn.CrossEntropyLoss(weight=class_weight.to(DEVICE))
optimizer = torch.optim.AdamW(network.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
print("langkah total:", total_steps)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 27124.22it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: ../pretrained/indobert
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


langkah total: 10580


In [20]:
import numpy as np
import torch
from tqdm import tqdm


def run_epoch(loader, training):
    network.train(training)
    total_loss, probabilities = 0.0, []
    for batch in tqdm(loader, leave=False):
        batch = batch.to(DEVICE)
        targets = batch.pop("labels")
        with torch.set_grad_enabled(training):
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
                logits = network(**batch).logits.float()
            loss = loss_function(logits, targets)
        if training:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        total_loss += loss.item() * len(targets)
        probability = torch.softmax(logits.detach(), dim=1)
        probabilities.append(probability.cpu().numpy())
    return total_loss / len(loader.dataset), np.concatenate(probabilities)


val_loss, val_probabilities = run_epoch(val_loader, training=False)
print("sebelum latih | val loss:", round(val_loss, 3), "| bentuk:", val_probabilities.shape)

sebelum latih | val loss: 1.129 | bentuk: (10578, 3)


In [21]:
import torch
from sklearn.metrics import f1_score

best_f1, waited = -1.0, 0
for epoch in range(EPOCHS):
    train_loss, _ = run_epoch(train_loader, training=True)
    val_loss, val_probabilities = run_epoch(val_loader, training=False)
    val_f1 = f1_score(part_val["target"], val_probabilities.argmax(axis=1), average="macro")
    print(f"epoch {epoch + 1}: train {train_loss:.3f} | val {val_loss:.3f} | F1 {val_f1:.4f}")
    if val_f1 > best_f1:
        best_f1, waited = val_f1, 0
        torch.save(network.state_dict(), "outputs/best_model.pt")
    else:
        waited += 1
    if waited >= PATIENCE:
        print("berhenti dini")
        break
print("F1 val terbaik:", round(best_f1, 4))

epoch 1: train 0.866 | val 0.783 | F1 0.5684


epoch 2: train 0.699 | val 0.725 | F1 0.6452


epoch 3: train 0.488 | val 0.773 | F1 0.6652


epoch 4: train 0.295 | val 0.949 | F1 0.6742
F1 val terbaik: 0.6742


In [22]:
import torch
from sklearn.metrics import classification_report

network.load_state_dict(torch.load("outputs/best_model.pt"))
_, val_probabilities = run_epoch(val_loader, training=False)
part_val = part_val.copy()
part_val["pred"] = np.array(CLASSES)[val_probabilities.argmax(axis=1)]
print(classification_report(part_val[TARGET], part_val["pred"], digits=4))
wrong = part_val[part_val[TARGET] != part_val["pred"]]
print(wrong[[TEXT, TARGET, "pred"]].head(6).to_string(max_colwidth=70))

              precision    recall  f1-score   support

     Negatif     0.6669    0.5807    0.6208      1648
      Netral     0.5995    0.6078    0.6036      3396
     Positif     0.7867    0.8103    0.7983      5534

    accuracy                         0.7095     10578
   macro avg     0.6843    0.6662    0.6742     10578
weighted avg     0.7079    0.7095    0.7081     10578

                                                               konten_berita sentimen_berita     pred
41458  Pekanbaru -\n\nPLN buka suara soal pemadaman lampu penerangan jala...         Positif  Negatif
15305  Jakarta (ANTARA) - Terdapat beberapa berita humaniora yang menarik...          Netral  Positif
31977  Jakarta (ANTARA) - Teknologi automasi dapat membantu meningkatkan ...         Positif   Netral
37506  JAKARTA - Forbes merilis daftar 50 orang terkaya di Korea Selatan ...         Positif   Netral
455    Jakarta (ANTARA) - Plt Juru Bicara KPK Ali Fikri menyatakan pihakn...         Negatif   Netral
46447  

In [28]:
from sklearn.metrics import f1_score
sample_submission = pd.read_csv("sample_submission.csv")

print("urutan id sama:", (test["id"].to_numpy() == sample_submission["id"].to_numpy()).all())
print("F1 macro test:", round(f1_score(test[TEXT], sample_submission[TARGET],
                                       average="macro"), 4))

urutan id sama: True
F1 macro test: 0.0
